In [ ]:
# %% [markdown]
# ============================================================
# UVS example — UnstructureVariableSelection (Quarterly)
# ============================================================
# Input: review-level CSV (raw reviews)
# Output: word-frequency matrix X (built internally), and mean rating y per document
#
# Key design choices:
# - Aggregate unit = locationId × quarter  (reduces document count vs monthly)
# - review_text = title + ". " + text     (NO injected metadata tokens)
# - Optional: filter out sparse documents (min reviews per doc)

# ============================================================
# Cell 1) Setup: paths + imports
# ============================================================

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- Find project root (folder containing "src") ---
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "251205_ParisHotel_comment_score.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("RAW_PATH:", RAW_PATH)

from bayesian_selection.unstructured import UnstructureVariableSelection
from bayesian_selection.word.word_corpus_cleaner import clean_word_corpus


PROJECT_ROOT: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection
SRC_DIR: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/src
RAW_PATH: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/data/raw/251205_ParisHotel_comment_score.csv


In [14]:
# ============================================================
# Cell 2) Load raw CSV
# ============================================================

df_raw = pd.read_csv(RAW_PATH)
print("df_raw shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
df_raw.head(3)

df_raw shape: (103885, 35)
Columns: ['id', 'status', 'reviewDetailPageWrapper', 'location', 'createdDate', 'publishedDate', 'userProfile', 'rating', 'publishPlatform', 'title', 'language', 'originalLanguage', 'text', 'username', 'connectionToSubject', 'mgmtResponse', 'locationId', 'helpfulVotes', 'labels', 'mcid', 'attribution', 'translationType', 'mtProviderId', 'alertStatus', 'roomTip', 'tripInfo', 'additionalRatings', 'photos', 'content', 'corpus', 'Mobility', 'Hospitality', 'PreRegister', 'CustomerService', 'Hygiene']


,id,status,reviewDetailPageWrapper,location,createdDate,publishedDate,userProfile,rating,publishPlatform,title,...,tripInfo,additionalRatings,photos,content,corpus,Mobility,Hospitality,PreRegister,CustomerService,Hygiene
0,1009363551,PUBLISHED,{'reviewDetailPageRoute': {'url': '/ShowUserRe...,"{'locationId': 23688737, 'name': 'Madame Rêve'...",2025-05-26,2025-05-26,"{'id': '3A81CFDE46465C21A9301854870A6728', 'is...",5.0,OTHER,Wonderful 5-star experience in the 1st. Perfe...,...,"{'stayDate': '2025-05-31', 'tripType': 'COUPLES'}","[{'rating': 5, 'ratingLabelLocalizedString': '...",[],Wonderful 5-star experience in the 1st. Perfe...,"['wonderful', 'experience', 'perfect', 'locate...",1.711189,0.983608,0.000000,-0.268319,0.491713
1,1041040518,PUBLISHED,{'reviewDetailPageRoute': {'url': '/ShowUserRe...,"{'locationId': 23688737, 'name': 'Madame Rêve'...",2025-12-01,2025-12-01,"{'id': '6A0D1121849E447CC7F3779491F2D4E9', 'is...",5.0,OTHER,Madame Reve ALL THE WAY!!!,...,"{'stayDate': '2025-11-30', 'tripType': 'COUPLES'}","[{'rating': 5, 'ratingLabelLocalizedString': '...",[],Madame Reve ALL THE WAY!!! We decided to go to...,"['way', 'paris', 'city', 'right', 'take', 'mom...",4.210150,-0.484083,-5.232259,-0.521082,-0.616493
2,1040950353,PUBLISHED,{'reviewDetailPageRoute': {'url': '/ShowUserRe...,"{'locationId': 23688737, 'name': 'Madame Rêve'...",2025-12-01,2025-11-30,"{'id': '6CA6F4B038B92A3CEA0042EA08C6DA8E', 'is...",3.0,MOBILE,Overpriced,...,"{'stayDate': '2025-11-30', 'tripType': 'COUPLES'}",[],"[{'photo': {'__typename': 'Media_PhotoResult',...",Overpriced My partner and I stayed in a Deluxe...,"['stay', 'room', 'staff', 'help', 'room', 'cle...",0.172877,0.000000,0.000000,0.141210,-1.255205


In [15]:
df_raw.columns

Index(['id', 'status', 'reviewDetailPageWrapper', 'location', 'createdDate',
       'publishedDate', 'userProfile', 'rating', 'publishPlatform', 'title',
       'language', 'originalLanguage', 'text', 'username',
       'connectionToSubject', 'mgmtResponse', 'locationId', 'helpfulVotes',
       'labels', 'mcid', 'attribution', 'translationType', 'mtProviderId',
       'alertStatus', 'roomTip', 'tripInfo', 'additionalRatings', 'photos',
       'content', 'corpus', 'Mobility', 'Hospitality', 'PreRegister',
       'CustomerService', 'Hygiene'],
      dtype='object')

In [17]:
# ============================================================
# Cell 3) Validate required columns
# ============================================================

required = ["locationId", "title", "text"]
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise KeyError(
        f"Missing required columns: {missing}\n"
        f"Available columns: {list(df_raw.columns)}"
    )

print("Required columns present:", required)


Required columns present: ['locationId', 'title', 'text']


In [18]:
# ============================================================
# Cell 4) Select rating/target column
# ============================================================

# Option A) Hardcode if you know the exact column name:
# RATING_COL = "comment_score"

# Option B) Auto-detect a numeric target column:
def pick_rating_col(df: pd.DataFrame) -> str:
    candidates = ["rating", "score", "stars", "overall", "label", "y", "comment_score"]
    for c in candidates:
        if c in df.columns:
            return c

    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        raise ValueError("No numeric columns found to use as rating/target.")
    non_null_ratio = {c: df[c].notna().mean() for c in num_cols}
    return max(non_null_ratio, key=non_null_ratio.get)

RATING_COL = pick_rating_col(df_raw)
print("Selected RATING_COL:", RATING_COL)


Selected RATING_COL: rating


In [19]:
# ============================================================
# Cell 5) Build UVS input df: locationId + (title + text) + rating
# ============================================================

df_u = df_raw[["locationId", "title", "text", RATING_COL]].copy()

df_u["locationId"] = df_u["locationId"].astype(str)

df_u["review_text"] = (
    df_u["title"].fillna("").astype(str).str.strip()
    + "\n\n"
    + df_u["text"].fillna("").astype(str).str.strip()
)

df_u["rating"] = pd.to_numeric(df_u[RATING_COL], errors="coerce")

# Minimal cleanup (intentionally light)
df_u["review_text"] = (
    df_u["review_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Drop unusable rows
df_u = df_u.dropna(subset=["rating"])
df_u = df_u[df_u["review_text"].str.len() > 0].reset_index(drop=True)

# Final UVS input schema
df_u = df_u.rename(columns={"locationId": "agg_id"})[["agg_id", "review_text", "rating"]]

print("df_u shape:", df_u.shape)
df_u.head(3)


df_u shape: (103885, 3)


,agg_id,review_text,rating
0,23688737.0,Wonderful 5-star experience in the 1st. Perfec...,5.0
1,23688737.0,Madame Reve ALL THE WAY!!! We decided to go to...,5.0
2,23688737.0,Overpriced My partner and I stayed in a Deluxe...,3.0


In [20]:
# ============================================================
# Cell 6) Initialize UnstructureVariableSelection
# ============================================================

selector = UnstructureVariableSelection(
    data=df_u,
    review_col="review_text",
    rating_col="rating",
    aggregate_col="agg_id",   # this is locationId in the raw data
    word_corpus_file=None,    # keep self-contained
    dist_type="correlate",    # simple + interpretable
    track=True,
)

print("UVS initialized.")

UVS initialized.


In [24]:
# ============================================================
# Cell 7) Generate (vocab + frequency matrices)
# ============================================================

TOP_N = 300
_ = selector.generate(top_n=TOP_N)

print(f"Generate complete. top_n={TOP_N}")



RAW_PATH: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/data/raw/251205_ParisHotel_comment_score.csv
OUT_DIR: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/output
300 of words are selected
Generate complete. top_n=300


In [ ]:
from BayesianVariableSelection.word.word_corpus_cleaner import clean_word_corpus

OUT_DIR = PROJECT_ROOT / "output"
OUT_DIR.mkdir(exist_ok=True)

WORD_CORPUS_XLSX = OUT_DIR / "example_unstructure_word_corpus.xlsx"
WORD_CORPUS_CLEANED = OUT_DIR / "example_unstructure_word_corpus_cleaned.xlsx"

selector.export_word_corpus(WORD_CORPUS_XLSX)
summary = clean_word_corpus(WORD_CORPUS_XLSX)

In [31]:
# ============================================================
# Cell 8) Initialize UnstructureVariableSelection
# ============================================================

selector = UnstructureVariableSelection(
    data=df_u,
    review_col="review_text",
    rating_col="rating",
    aggregate_col="agg_id",   # this is locationId in the raw data
    word_corpus_file=WORD_CORPUS_CLEANED,    # keep self-contained
    dist_type="correlate",    # simple + interpretable
    track=True,
)

# Generate frequency matrix + mean rating per aggregate unit
_ = selector.generate()

207 of words are selected


In [32]:
# ---- START SMALL for debugging ----
SEGMENTS = 2       # aka segment_no (latent segments K)
ITER     = 1000    # MCMC iterations
BURNIN   = 1000    # burn-in draws / default is 2000
STEP     = 1       # save every STEP draws
CUTOFF   = 0.70    # Correlate: cutoff = 0.7 / Cooccur: cutoff = 0.3

selector.initialize(
    segment_no=SEGMENTS,
    iteration=ITER,
    burnin=BURNIN,
    cutoff=CUTOFF,
    step=STEP
)
selector.fit()

  0%|          | 0/2000 [00:00<?, ?it/s]

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning:

invalid value encountered in divide

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning:

invalid value encountered in divide



In [34]:
selector.visualize_topic_seg(0, k = 3, cut = 0.9).update_layout(width=1200, height=600, xaxis_tickangle = 45)

In [ ]:
np.unique(selector.find_membership(), return_counts=True)

In [ ]:
selector.find_appropriate_document('appreciate')

In [ ]:
selector.diagnose_model()

In [ ]:
selector.find_correlation_by_seg(0)

In [ ]:
freq = selector.frequency_matrix.copy()
freq['membership'] = selector.find_membership()